In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def load_and_preprocess_data(georock_path='./data/raw/georock-data.csv', caio_path='./data/raw/Results_Caio.xlsx'):
        """Load and preprocess the raw geochemical data."""
        print("Loading and preprocessing raw data...")
        
        # Load GEOROCK data with better memory handling
        df = pd.read_csv(georock_path, encoding='latin1', low_memory=False)
        df = df[~df['ALTERATION'].str.contains('ALTERED', na=False)]
        
        # Filter by material types and create proper copies
        IT_VOLCANIC_GLASS = df[df['TYPE OF MATERIAL'].str.contains("VOLCANIC GLASS", na=False) == True].copy()
        IT_VOLCANIC_WR = df[df['TYPE OF MATERIAL'].str.contains("WHOLE ROCK", na=False) == True].copy()
        IT_VOLCANIC_INCLUSIONS = df[df['TYPE OF MATERIAL'].str.contains("INCLUSION", na=False) == True].copy()
        
        dataframes = [IT_VOLCANIC_GLASS, IT_VOLCANIC_WR, IT_VOLCANIC_INCLUSIONS]
        
        # Clean column names
        for df_subset in dataframes:
            df_subset.columns = [c.replace('(', '').replace(')', '').replace('%', '')
                               .replace(' ', '').replace('.', '') for c in df_subset.columns]
            df_subset.loc[:, 'Latitude'] = (df_subset['LATITUDEMAX'] + df_subset['LATITUDEMIN']) / 2
            df_subset.loc[:, 'Longitude'] = (df_subset['LONGITUDEMAX'] + df_subset['LONGITUDEMIN']) / 2
        
        # Further column cleaning
        for df_subset in dataframes:
            replacements = {
                'WT': '', 'PPM': '', 'SR': 'Sr', 'TH': 'Th', 'NB': 'Nb', 'TA': 'Ta',
                'LA': 'La', 'CE': 'Ce', 'ND': 'Nd', 'ZR': 'Zr', 'HF': 'Hf', 'SM': 'Sm',
                'GD': 'Gd', 'YB': 'Yb', 'LU': 'Lu', 'TI': 'Ti', 'RB': 'Rb', 'BA': 'Ba',
                'TiO2': 'TIO2', 'SiO2': 'SIO2', 'Al2O3': 'AL2O3', 'CaO': 'CAO',
                'MgO': 'MGO', 'MnO': 'MNO', 'Na2O': 'NA2O', 'LOCATiON': 'LOCATION'
            }
            for old, new in replacements.items():
                df_subset.columns = [c.replace(old, new) for c in df_subset.columns]
        
        # Combine dataframes with proper index handling
        dataTOTF = pd.concat(dataframes, ignore_index=True)
        
        # Create a clean copy to avoid fragmentation issues
        dataTOTF = dataTOTF.copy()
        
        # Calculate FE2O3Tfinal more efficiently
        dataTOTF['FE2O3Tfinal'] = np.nan
        
        # Method 1: From FEOT
        mask1 = dataTOTF['FEOT'].notna()
        dataTOTF.loc[mask1, 'FE2O3Tfinal'] = dataTOTF.loc[mask1, 'FEOT'] / 0.8998
        
        # Method 2: From FE2O3 + FEO
        mask2 = dataTOTF['FE2O3Tfinal'].isna() & dataTOTF['FE2O3'].notna() & dataTOTF['FEO'].notna()
        dataTOTF.loc[mask2, 'FE2O3Tfinal'] = dataTOTF.loc[mask2, 'FE2O3'] + dataTOTF.loc[mask2, 'FEO'] / 0.8998
        
        # Method 3: Direct from FE2O3T
        mask3 = dataTOTF['FE2O3Tfinal'].isna() & (dataTOTF['FE2O3T'] >= 0)
        dataTOTF.loc[mask3, 'FE2O3Tfinal'] = dataTOTF.loc[mask3, 'FE2O3T']
        
        # Calculate sum of traces
        trace_cols = ['Nb', 'Zr', 'La', 'Ce', 'Sr', 'Ba', 'Rb']
        dataTOTF['SumTracesPercent'] = dataTOTF[trace_cols].sum(axis=1) / 10000
        
        # Calculate total weight and filter
        column_list = ['SIO2', 'TIO2', 'AL2O3', 'FE2O3Tfinal', 'CAO', 'MGO', 'MNO', 
                      'K2O', 'NA2O', 'P2O5', 'SumTracesPercent']
        dataTOTF['TotalWT'] = dataTOTF[column_list].sum(axis=1)
        dataTOTF = dataTOTF[(dataTOTF['TotalWT'] >= 94) & (dataTOTF['TotalWT'] <= 103)].copy()
        
        # Normalize compositions efficiently
        normalization_mapping = {
            'SIO2N': 'SIO2', 'TIO2N': 'TIO2', 'AL2O3N': 'AL2O3', 'FE2O3TN': 'FE2O3Tfinal',
            'CAON': 'CAO', 'MGON': 'MGO', 'MNON': 'MNO', 'NA2ON': 'NA2O', 'K2ON': 'K2O', 
            'P2O5N': 'P2O5'
        }
        
        # Create all normalized columns at once
        normalized_data = {}
        for norm_col, orig_col in normalization_mapping.items():
            normalized_data[norm_col] = dataTOTF[orig_col] / dataTOTF['TotalWT']
        
        # Normalize trace elements
        for element in trace_cols:
            normalized_data[f'{element}N'] = dataTOTF[element] / (10000 * dataTOTF['TotalWT'])
        
        # Add total alkali
        normalized_data['TotalAlkali'] = normalized_data['K2ON'] + normalized_data['NA2ON']
        
        # Assign all normalized columns at once
        for col, values in normalized_data.items():
            dataTOTF[col] = values
        
        # Process volcanic regions
        volcanic_regions = _process_volcanic_regions(dataTOTF)
        dataTOTF = pd.concat(volcanic_regions, ignore_index=True)
        dataTOTF = dataTOTF[(dataTOTF['SIO2N'] >= 0.4) & (dataTOTF['SIO2N'] <= 0.8)].copy()
        
        # Load and process CAIO data
        CAIO = pd.read_excel(caio_path)
        print(f"Loaded CAIO data: {len(CAIO)} samples")
        
        # CAIO samples are the unknown samples we want to classify
        CAIO['lettercode'] = "Caio"  # Set BEFORE processing
        CAIO = _process_caio_data(CAIO)
        
        print(f"Processed CAIO data: {len(CAIO)} samples")
        print(f"CAIO lettercode values: {CAIO['lettercode'].unique()}")
        
        # Select only the columns that both datasets have
        common_columns = ['SIO2N', 'TIO2N', 'AL2O3N', 'FE2O3TN', 'CAON', 'MGON', 'MNON', 
                         'NA2ON', 'K2ON', 'P2O5N', 'NbN', 'ZrN', 'LaN', 'CeN', 'SrN', 
                         'BaN', 'RbN', 'lettercode']
        
        # Ensure both dataframes have the same columns
        dataTOTF_filtered = dataTOTF[common_columns].copy()
        CAIO_filtered = CAIO[common_columns].copy()
        
        print(f"DataTOTF filtered: {len(dataTOTF_filtered)} samples")
        print(f"DataTOTF lettercode values: {dataTOTF_filtered['lettercode'].unique()}")
        print(f"CAIO filtered: {len(CAIO_filtered)} samples") 
        print(f"CAIO filtered lettercode values: {CAIO_filtered['lettercode'].unique()}")
        
        # Combine datasets with proper index handling
        final_data = pd.concat([dataTOTF_filtered, CAIO_filtered], ignore_index=True)
        
        print(f"Combined data: {len(final_data)} samples")
        print(f"Combined lettercode values: {final_data['lettercode'].value_counts()}")
        
        # Clean final data
        for col in final_data.select_dtypes(include=['number']).columns:
            final_data[col] = pd.to_numeric(final_data[col], errors='coerce')
        
        # Remove rows where any numeric values are NaN, but be more lenient with zeros
        # Some trace elements can legitimately be zero or very small
        numeric_cols = final_data.select_dtypes(include=['float64', 'int64']).columns
        essential_cols = ['SIO2N', 'TIO2N', 'AL2O3N', 'FE2O3TN', 'CAON', 'MGON', 'NA2ON', 'K2ON']  # Major elements that shouldn't be zero
        
        before_cleaning = len(final_data)
        
        # Remove rows with NaN values
        final_data = final_data.dropna(subset=numeric_cols)
        after_nan_removal = len(final_data)
        
        # Only require essential major elements to be positive, allow trace elements to be small/zero
        final_data = final_data[(final_data[essential_cols] > 0).all(axis=1)].copy()
        
        # For trace elements, just remove clearly invalid negative values
        trace_cols = ['NbN', 'ZrN', 'LaN', 'CeN', 'SrN', 'BaN', 'RbN']
        final_data = final_data[(final_data[trace_cols] >= 0).all(axis=1)].copy()  # >= 0 instead of > 0
        
        after_cleaning = len(final_data)
        
        print(f"Cleaning steps:")
        print(f"  After NaN removal: {after_nan_removal} samples (removed {before_cleaning - after_nan_removal})")
        print(f"  After major element filter: {after_cleaning} samples (removed {after_nan_removal - after_cleaning})")
        print(f"Final lettercode values: {final_data['lettercode'].value_counts()}")
        
        # Debug: Show some statistics about the CAIO samples specifically
        caio_samples = final_data[final_data['lettercode'] == "Caio"]
        if len(caio_samples) > 0:
            print(f"\nCAIO samples in final data: {len(caio_samples)}")
        else:
            print(f"\nNo CAIO samples in final data. Let's check what happened...")
            # Check what values the CAIO samples had before cleaning
            caio_before_cleaning = pd.concat([dataTOTF_filtered, CAIO_filtered], ignore_index=True)
            caio_before = caio_before_cleaning[caio_before_cleaning['lettercode'] == "Caio"]
        
        return final_data
    
def _process_volcanic_regions(dataTOTF):
        """Process and assign control codes to volcanic regions."""
        regions_config = {
            "VV": ["VESUVIUS"],
            "EV": ["ETNA"], 
            "PF": ["ISCHIA", "FLEGREI", "PROCIDA"],
            "AI": ["AEOLIAN"],
            "RMP": ["LATERA", "MONTEFIASCONE", "BOLSENA", "VICO", "SABATINI", "ALBAN"],
            "TMP": ["VINCENZO", "ROCCASTRADA", "TOLFA", "CIMINO", "AMIATA", "GIGLIO", "CAPRAIA", "MONTECATINI", "ORCIATICO", "RADICOFANI", "ALFINA", "CECINA"],
            "PI": ["PANTELLERIA"],
            "IAVP": ["VENANZO", "POLINO", "CUPAELLO"],
            "MV": ["VULTURE"],
            "ERP": ["ROCCAMONFINA", "MONTI ERNICI"]
        }
        
        region_dataframes = []
        for letter_code, locations in regions_config.items():
            for location in locations:
                location_key = "MONTI ERNICI" if location == "MONTI ERNICI" else location
                region_df = dataTOTF[dataTOTF['LOCATION'].str.contains(location_key, na=False) == True].copy()
                region_df['lettercode'] = letter_code
                region_dataframes.append(region_df)
        
        return region_dataframes
    
def _process_caio_data(CAIO):
        """Process CAIO data to match the main dataset format."""
        CAIO = CAIO.copy()  # Create a proper copy
        
        # IMPORTANT: CAIO samples are the UNKNOWN samples (lettercode="Caio")
        CAIO['lettercode'] = "Caio"
        
        # Calculate sum of traces
        trace_elements = ['Nb', 'Zr', 'La', 'Ce', 'Sr', 'Ba', 'Rb']
        CAIO['SumTracesPercent'] = CAIO[trace_elements].sum(axis=1) / 10000
        
        column_list = ['SIO2', 'TIO2', 'AL2O3', 'FE2O3T', 'CAO', 'MGO', 'MNO', 
                      'K2O', 'NA2O', 'P2O5', 'SumTracesPercent']
        CAIO['TotalWT'] = CAIO[column_list].sum(axis=1)
        CAIO = CAIO[(CAIO['TotalWT'] >= 94) & (CAIO['TotalWT'] <= 103)].copy()
        
        # Normalize CAIO data efficiently
        normalization_mapping = {
            'SIO2N': 'SIO2', 'TIO2N': 'TIO2', 'AL2O3N': 'AL2O3', 'FE2O3TN': 'FE2O3T',
            'CAON': 'CAO', 'MGON': 'MGO', 'MNON': 'MNO', 'NA2ON': 'NA2O', 'K2ON': 'K2O', 
            'P2O5N': 'P2O5'
        }
        
        # Create normalized columns
        normalized_data = {}
        for norm_col, orig_col in normalization_mapping.items():
            normalized_data[norm_col] = CAIO[orig_col] / CAIO['TotalWT']
        
        # Normalize trace elements
        for element in trace_elements:
            normalized_data[f'{element}N'] = CAIO[element] / (10000 * CAIO['TotalWT'])
        
        # Add total alkali
        normalized_data['TotalAlkali'] = normalized_data['K2ON'] + normalized_data['NA2ON']
        
        # CRITICAL: Ensure lettercode="Caio" for unknown samples
        normalized_data['lettercode'] = "Caio"
        
        # Create final dataframe with only the columns we need
        final_cols = ['SIO2N', 'TIO2N', 'AL2O3N', 'FE2O3TN', 'CAON', 'MGON', 'MNON', 
                     'NA2ON', 'K2ON', 'P2O5N', 'NbN', 'ZrN', 'LaN', 'CeN', 'SrN', 
                     'BaN', 'RbN', 'lettercode']
        
        result_df = pd.DataFrame(normalized_data)[final_cols]
        
        return result_df

In [ ]:
data = load_and_preprocess_data(georock_path='../data/raw/georock-data.csv', caio_path='../data/raw/Results_Caio.xlsx')

In [ ]:
len(data)

In [ ]:
data['lettercode'].value_counts()

In [ ]:
data.to_csv('../data/processed/caio_italy_benchmark/full_italian_data.csv')